# QuantSwing AI — générateur de chandeliers historiques

Ce carnet prépare **un seul fichier CSV** contenant plusieurs actions, utilisable dans l’onglet **Laboratoire des preuves** de QuantSwing AI sur iPad.

- Aucun compte de courtage ni donnée personnelle n'est nécessaire.
- Source gratuite non officielle : Yahoo Finance via `yfinance`, avec disponibilité et droits d’utilisation susceptibles de varier.
- Les données peuvent être incomplètes, corrigées ou ajustées. Elles ne suffisent pas à prouver qu’une stratégie fonctionne dans le futur.
- En cas d'erreur de téléchargement, réessaie avec moins de symboles ou utilise un CSV provenant d'une source autorisée.

**Utilisation sur iPad :** Exécution → Tout exécuter, puis importer `QuantSwing_historiques.csv` dans le Laboratoire de QuantSwing AI.


In [ ]:
# Installer les dépendances gratuites sur l'environnement Colab.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'yfinance'])
print('Bibliothèque installée.')


In [ ]:
# Modifie librement la sélection. .TO indique une action cotée au TSX.
TICKERS = ['NVDA', 'AMD', 'AAPL', 'MSFT', 'SHOP.TO', 'CCO.TO', 'WPM.TO', 'ENB.TO']
PERIODE = '5y'  # 5y, 2y, 1y ; idéalement 5y pour les premiers backtests
print('Actions sélectionnées :', ', '.join(TICKERS))


In [ ]:
import csv
import io
import time
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import yfinance as yf
from google.colab import files

rows = []
failures = []
symbols = list(dict.fromkeys(str(x).strip().upper() for x in TICKERS if str(x).strip()))
if not 1 <= len(symbols) <= 60:
    raise ValueError('Sélectionne entre 1 et 60 actions.')

for symbol in symbols:
    try:
        df = yf.Ticker(symbol).history(period=PERIODE, interval='1d', auto_adjust=True, actions=False)
        if df is None or df.empty:
            raise ValueError('Aucun historique disponible')
        cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        if not all(col in df.columns for col in cols):
            raise ValueError('Colonnes OHLCV manquantes')
        df = df[cols].dropna().copy()
        # Éviter de traiter comme terminée une séance du jour encore incomplète.
        now_toronto = datetime.now(ZoneInfo('America/Toronto'))
        if now_toronto.hour < 18:
            df = df[df.index.date < now_toronto.date()]
        for dt, point in df.iterrows():
            o, hi, lo, close, volume = (float(point[k]) for k in cols)
            if min(o, hi, lo, close) <= 0 or volume < 0:
                continue
            rows.append([symbol, dt.strftime('%Y-%m-%d'), o, hi, lo, close, int(volume)])
        print(f'{symbol} : {len(df)} séances brutes')
    except Exception as e:
        failures.append((symbol, str(e)[:250]))
        print(f'{symbol} : échec ({str(e)[:120]})')
    time.sleep(0.4)

if not rows:
    raise RuntimeError('Aucune donnée reçue : contrôle les symboles ou le fournisseur.')
output = 'QuantSwing_historiques.csv'
with open(output, 'w', encoding='utf-8', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['Symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    writer.writerows(rows)
print(f'\nFichier prêt : {output} — {len(rows)} chandeliers, {len(set(r[0] for r in rows))} actions')
if failures:
    print('Actions non obtenues :', failures)
print('Données potentiellement ajustées ; vérifier l’absence de biais de survivance et les événements corporatifs.')
files.download(output)
